# PPO on CartPole-v1

**Proximal Policy Optimization (PPO)** is a popular on-policy algorithm that constrains the policy update step using a clipped surrogate objective:

$$L^{CLIP}(\theta) = \mathbb{E}_t\left[\min\left(r_t(\theta)\hat{A}_t,\;\text{clip}(r_t(\theta),\,1-\epsilon,\,1+\epsilon)\hat{A}_t\right)\right]$$

where $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$ is the probability ratio.

> Paper: [Proximal Policy Optimization Algorithms](https://arxiv.org/abs/1707.06347)

In this notebook we train PPO on the classic **CartPole-v1** task using `lit-sb3`.


## Setup


In [ ]:
import warnings

warnings.filterwarnings("ignore")

import gymnasium as gym
import numpy as np

from lit_sb3 import RLEnvDataModule, RLTrainer
from lit_sb3.algorithms.ppo.ppo import PPO

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir ./

## Create the model

Key hyperparameters:

| Parameter    | Description                                                     |
| ------------ | --------------------------------------------------------------- |
| `n_steps`    | Steps per env per rollout (rollout buffer = `n_steps × n_envs`) |
| `batch_size` | Mini-batch size for the surrogate loss                          |
| `n_epochs`   | Passes over the rollout buffer per update                       |
| `clip_range` | Clipping epsilon $\epsilon$ for the surrogate objective         |
| `gae_lambda` | GAE $\lambda$ for advantage estimation                          |

In `lit-sb3`, the environment is attached by `RLTrainer.fit(...)`, so the concrete `model.policy` module is built during training setup rather than at constructor time.


In [ ]:
model = PPO(
    "MlpPolicy",
    verbose=1,
    n_steps=256,
    batch_size=64,
    n_epochs=10,
    learning_rate=3e-4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.0,
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs={"net_arch": {"pi": [64, 64], "vf": [64, 64]}},
)
print(model.policy_class.__name__)

## Train

Use the explicit `RLTrainer` entry point so trainer configuration and rollout-driven training stay in one place.


In [ ]:
datamodule = RLEnvDataModule(train_env="CartPole-v1")
trainer = RLTrainer(max_steps=25_000)
trainer.fit(model, datamodule=datamodule)
print(model.policy)

## Evaluate


In [ ]:
eval_results = trainer.test(
    env=gym.make("CartPole-v1"),
    n_eval_episodes=10,
)
eval_results[0]